In [ ]:
import os
import shutil
import boto3
from tqdm import tqdm
from datetime import datetime, timedelta

# Set the environment variables
# use your AWS credentials insted of these
os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''

In [ ]:
class S3DataDownloader:
    '''
    Class to download data from S3 bucket in the format of date/fish_name/fresh_type
    '''
    
    def __init__(self, bucket_name, download_path):
        """
        Initializes the S3DataDownloader instance.

        Args:
            bucket_name (str): The name of the S3 bucket.
            download_path (str): The local directory to download files to.
        """
        self.bucket_name = bucket_name
        self.download_path = download_path
        # Create an S3 client
        self.s3_client = boto3.client('s3')
        self.grouped_objects = self.group_objects_by_date()

    def list_s3_objects(self, bucket_name=None):
        """
        Lists all objects in the specified S3 bucket.

        Args:
            bucket_name (str): The name of the S3 bucket.

        Returns:
            list: A list of objects in the S3 bucket.
        """
        if bucket_name is None:
            bucket_name = self.bucket_name
        object_list = []
        # Use a paginator to iterate through all the objects in the bucket
        paginator = self.s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket_name)

        for page in page_iterator:
            # Get the list of objects in the current page
            objects = page.get('Contents', [])
            object_list.extend(objects)
        print("\nlist_s3_objects done")
        return object_list

    def group_objects_by_date(self):
        """
        Lists all objects in the S3 bucket & groups them by their last modified date.

        Returns:
            dict: A dictionary where keys are dates and values are lists of object keys.
        """
        objects = self.list_s3_objects()
        grouped_objects = {}
        for obj in objects:
            key = obj['Key']

            # extracting date from key
            date = (key.split('/')[-1])[:8]
            date = date[:4] + '-' + date[4:6] + '-' + date[6:8]

            # extracting date from LastModified
            # timestamp = obj['LastModified']
            # date = str(timestamp.date())

            if date in grouped_objects:
                grouped_objects[date].append(key)
            else:
                grouped_objects[date] = [key]

        # Sort the dictionary based on date values in the keys
        grouped_objects = dict(sorted(grouped_objects.items(),
                                      key=lambda item: item[0]))
        print("group_objects_by_date done\n")
        return grouped_objects

    def replace_misspelled_folder_names(self, species_name):
        misspelled_folders = {
            'Are' : ['Ar', 'Are'],
            'Basa' : ['Basa', 'Basaa'],
            'Barracuda' : ['Barcoda', 'Barkoda', 'Barracoda', 'Barracuda'],
            'Bolo' : ['Bolo', 'Bulo'],
            'Catla' : ['Katala', 'Katalaa', 'Katla'],
            'Croaker' : ['Kokor', 'Croaker', 'Silver croaker'],
            'Chara pona' : ['Chara pana'],
            'Demo' : ['Demo', 'Demo2', 'Test', 'Trial'],
            'Emperor' : ['Comprel', 'Emperor', 'Emporwel', 'Empowel',
                         'M perl', 'M preal'],
            'Hilsa' : ['Hilsa', 'Hilis', 'Hilisa'],
            'Lady' : ['Lady', 'Ledi'],
            'Malabar trevally' : ['Mabar tavili', 'Malbhot', 'Travely', 'Travaily',
                                  'Trvili', 'Trevally', 'Travelly', 'Giant trevally'],
            'Needle' : ['Needale', 'Nidal', 'Nidil'],
            'Parsi' : ['Parci'],
            'Pearl spot' : ['(bloch,', 'Bloch,', 'Bloch',
                            'Pearl spot', 'Pearls spot', 'Hols spot',
                            'Green chromide', 'Green chormide',
                            'Hals spot'],
            'Sea bass' : ['C boss', 'C boos', 'Siba'],
            'Shol' : ['Sholo'],
            'Snapper' : ['Sinper', 'Sniper'],
            'White snapar' : ['White snapper'],
        }

        for key, misspellings in misspelled_folders.items():
            if species_name in misspellings:
                return key
        return species_name

    def download_data(self, date, keys):
        # Create a directory for the date if it doesn't exist
        date_directory = os.path.join(self.download_path, date)
        os.makedirs(date_directory, exist_ok=True)

        # Download each object in the group
        for key in tqdm(keys, desc=f'Downloading {date} data'):
            # Extract fish name and fresh type from the image name
            image_name = key.split('/')[-1]
            try:
                fish_name, fresh_type = image_name.split('_')[-2:]
            except:
                continue
            fresh_type = fresh_type.split('.')[0]
            fish_name = fish_name.capitalize()
            fresh_type = fresh_type.capitalize()

            if fish_name.endswith(" "):
                fish_name = fish_name[:-1]
            if fresh_type.endswith(" "):
                fresh_type = fresh_type[:-1]

            old_fish_name = fish_name

            fish_name = self.replace_misspelled_folder_names(fish_name)

            # if fish_name == "Demo":
            #     continue

#             if fish_name not in ['Sardine', 'Mackerel', 'White prawns']:
#                 continue

            if fish_name not in ('Sardine'):
                continue

            if old_fish_name != fish_name:
                duplicate_folder = os.path.join(date_directory, old_fish_name)
                if os.path.exists(duplicate_folder):
                    shutil.rmtree(duplicate_folder)

            # Create a directory structure:
            # date_folder/fish_name_folder/fresh_type_folder
            fish_directory = os.path.join(date_directory, fish_name)
            fresh_directory = os.path.join(fish_directory, fresh_type)
            
            os.makedirs(fresh_directory, exist_ok=True)
            file_name = os.path.join(fresh_directory, image_name)

            # Download the object if it doesn't already exist locally
            if not os.path.exists(file_name):
                self.s3_client.download_file(self.bucket_name, key, file_name)
            else:
                pass
                # print(f"Skipped (already exists): {file_name}")

    def download_daily_data(self):
        for date, keys in self.grouped_objects.items():
            self.download_data(date, keys)

    def download_specific_date_data(self, specific_date):
        valid_dates = [date for date in self.grouped_objects.keys()]
        print(valid_dates)
        if specific_date not in valid_dates:
            print(f"Data is not collected on : {specific_date}")
            return
        keys = self.grouped_objects[specific_date]
        self.download_data(specific_date, keys)

    def download_weekly_data(self, start_date, end_date):
        for date, keys in self.grouped_objects.items():
            if start_date <= date <= end_date:
                self.download_data(date, keys)

In [ ]:
# Example usage
bucket_name='fish-data-collection-v2'
download_path="/kaggle/working/S3 Daily Data"

data_downloader=S3DataDownloader(bucket_name, download_path)

# Download all data daily
# data_downloader.download_daily_data()

# Date should be in YYYY-MM-DD format

# # Download data for a specific date
# specific_date = '2023-05-23'
# data_downloader.download_specific_date_data(specific_date)

# Download data weekly between two dates
start_date = '2024-10-09'
end_date = '2024-11-03'
data_downloader.download_weekly_data(start_date, end_date)

Combining S3 Daily Data

In [ ]:
def generate_date_list(start_date, source_dir):
    """
    Generate a list of dates from a specific start date to the end date.

    Parameters:
        start_date (str): Start date in the format 'YYYY-MM-DD'.
        source_dir (str): The source directory containing data organized by dates.

    Returns:
        list: List of dates in the format 'YYYY-MM-DD'.

    """
    all_dates = sorted(os.listdir(source_dir))
    if start_date is None:
        return all_dates

    # list of date folders after the given date in the folder
    dates_list = []
    for date in all_dates:
        if date >= start_date:
            dates_list.append(date)
    return dates_list


def organize_files(source_dir, destination_dir, start_date=None):
    """
    Organize files from source directory to destination directory based on dates in the format of destination_dir/fish_name/fresh_type.

    Parameters:
        source_dir (str): The source directory containing data organized by dates.
        destination_dir (str): The destination directory to organize the data.
        dates_list (list): List of dates to organize. If None, all dates will be organized.

    """
    os.makedirs(destination_dir, exist_ok=True)
    dates_list = generate_date_list(start_date, source_dir)
    for date in dates_list:
        date_path = os.path.join(source_dir, date)
        if not os.path.isdir(date_path): continue

        for species in tqdm(os.listdir(date_path),
                            desc=f'Loading {date} data'):
            src_species_path = os.path.join(date_path, species)
            des_species_path = os.path.join(destination_dir, species)
            if not os.path.isdir(src_species_path): continue

            for fresh_type in os.listdir(src_species_path):
                src_fresh_type_path = os.path.join(src_species_path, fresh_type)
                des_fresh_type_path = os.path.join(des_species_path, fresh_type)
                if not os.path.isdir(src_fresh_type_path): continue
                os.makedirs(des_fresh_type_path, exist_ok=True)

                for img in os.listdir(src_fresh_type_path):
                    img_path = os.path.join(src_fresh_type_path, img)
                    des_filepath = os.path.join(des_fresh_type_path, img)
                    if not os.path.exists(des_filepath):
                        shutil.copy(img_path, des_filepath)

In [ ]:
source_directory = "S3 Daily Data"
destination_directory = "Final Data"

# For all dates
start_date_str = None
organize_files(source_directory, destination_directory, start_date_str)

# # For dates starting from a specific date
# start_date_str = '2024-09-23'  # Replace with your desired start date
# organize_files(source_directory, destination_directory, start_date_str)

Creating segmented Dataset

In [ ]:
! pip install ultralytics

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
import cv2
def get_segmented_img(pred, white_background=False, index=0):
    img_shape = pred.orig_shape
    img = pred.orig_img.copy()
#     img = cv2.imread(img_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img_shape[0], img_shape[1]
    masks = pred.masks.xy
    seg_img_list = []
    for mask in masks:
#         mask = masks[index]
        mask_points = mask.astype(int)
        binary_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(binary_mask, [mask_points], 255)
        masked_img = cv2.bitwise_and(img, img, mask=binary_mask)
        x, y, w, h = cv2.boundingRect(mask_points)
        x1, y1, x2, y2 = (x, y, x+w, y+h)
        box = [x1, y1, x2, y2]
        segment_image = np.zeros((height, width, 3), dtype=np.uint8)
        segment_image[y:y+h, x:x+w] = masked_img[y:y+h, x:x+w]
        segment_image = segment_image[y:y+h, x:x+w]
        # white_background = False
        if white_background:
            black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
            segment_image[black_mask] = [255, 255, 255]
        seg_img_list.append(segment_image)
    return seg_img_list


def segment_img(img_path, seg_model):
    results = seg_model(img_path, imgsz=640, conf=0.75, iou=0.7, verbose=False)
    if not results[0].masks: return None
    seg_img_list = get_segmented_img(results[0])
#     return seg_img_list[0]
    return seg_img_list

In [ ]:
from ultralytics import YOLO
seg_model = YOLO('fish.pt')

l = segment_img('Final Data/Sardine/Good/20241017121715256_sardine_good.jpeg', seg_model)
plt.imshow(l[0]);
plt.axis('off');

In [ ]:
src_dir = 'Final Data/Sardine'
des_dir = 'sardine_segmented_dataset' 
os.makedirs(des_dir, exist_ok=True)

for f in os.listdir(src_dir):
    folder_path = os.path.join(src_dir, f)
    new_folder_path = os.path.join(des_dir, f)
    os.makedirs(new_folder_path, exist_ok=True)
    l = [os.path.join(folder_path, img) for img in os.listdir(folder_path)]
    for img_path in tqdm(l, desc=f):
        seg_img_list = segment_img(img_path, seg_model)
        if seg_img_list is None: 
#             print(img_path, 'is not segmented')
            continue
        for i, seg_img in enumerate(seg_img_list):
            img_name = img_path.split('/')[-1]
            img_name = img_name.split('.')[0]+f'_({i})'+'.jpg'   ############ we should check if jpg or png
            filepath = os.path.join(new_folder_path, img_name)
#             if os.path.exists(filepath): continue
            cv2.imwrite(filepath, seg_img)